In [ ]:
#resblock description

#input ((->norm relu - > conv 3 3 3 -> norm relu -> conv 1 1 1 -> norm relu) + positional encoding -> axial attention )+ input

In [ ]:
#config for vqvae
#input size 16 X 64 X 64

#latent size 4 x 32 x 32
#b commitment loss -> 0.25
#batch size 32
#learning rate - 7*10^(-4)
#hidden units -> 240

#residual units ->128
#residual layers ->4
#uses attention -> Yes

#codebook size ->1024
#codebook dimension ->256
#encoder filter size ->3
#upsampling filter size ->4
#training steps -> 100k

#fsq is used instead of vq-vae

In [ ]:
#config for transformer over vqvae prior
#input size 4 x 32 x 32
#batch size 32
#learning rate 3*10^(-4)
#vocabulary size 1024
#attention heads 8
#attention layers 20
#embedding 1024
#feedforward hidden size 4096
#resnet depth n/a
#resnet unit n/a
#dropout 0.2
#training steps 200k/600k


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import torchvision
import torchvision.datasets as datasets
import torchvision.utils as utils

In [ ]:
class three_dimensional_Attention(nn.Module):
  def __init__(self, n_hiddens, n_heads):
    super().__init__()

    kwargs=dict(shape=(0,)*3, dim_q=n_hiddens, dim_kv=n_hiddens,
                      n_head=n_head, n_layer=1, causal=False, attn_type='axial')
    self.attn_w=nn.MultiHeadAttention(attn_kwargs=dict(axial_dim=-2), **kwargs)
    self.attn_h=nn.MultiHeadAttention(attn_kwargs=dict(axial_dim=-3), **kwargs)
    self.attn_t=nn.MultiHeadAttention(attn_kwargs=dict(axial_dim=-4), **kwargs)

  def forward(self, x):
    x = shift_dim(x, 1, -1)
    x = self.attn_w(x, x, x) + self.attn_h(x, x, x) + self.attn_t(x, x, x)
    x = shift_dim(x, -1, 1)
    return x



In [ ]:
class VQ_VAE_ResBlock(nn.Module):
  def __init__(self, input_channels):
    super().__init__()
    self.layer_norm1=nn.BatchNorm3d(input_channels)
    self.relu1=nn.ReLU()
    self.conv1=nn.Conv3d(input_channels, input_channels//2, kernel_size=(3,3,3), stride=(1,1,1), padding=(1,1,1), bias=False)
    self.layer_norm2=nn.BatchNorm3d(input_channels//2)
    self.relu2=nn.ReLU()
    self.conv2=nn.Conv3d(input_channels//2, input_channels, kernel_size=(1,1,1), stride=(1,1,1), padding=0, bias=False)
    self.layer_norm3=nn.BatchNorm3d(input_channels)
    self.relu3=nn.ReLU()
    #attention layers is needed
  def forward(self, x):
    h=self.layer_norm1(x)
    h=self.relu1(h)
    h=self.conv1(h)
    h=self.layer_norm2(h)
    h=self.relu2(h)
    h=self.conv2(h)
    h=self.layer_norm3(h)
    h=self.relu3(h)
    h=three_dimensional_Attention(h)

    h=h+x
    return h

In [ ]:
class Encoder(nn.Module):
  def __init__(self, out_channels):
    super().__init__()
    self.conv1=nn.Conv3d(3, out_channels, kernel_size=(3,3,3), stride=(2,2,2), padding=(1,1,1), bias=True)
    self.conv2=nn.Conv3d(out_channels, out_channels, kernel_size=(3,3,3), stride=(2,1,1), padding=(1,1,1), bias=True)
    self.conv3=nn.Conv3d(out_channels, out_channels, kernel_size=(3,3,3), stride=(1,1,1)padding=(1,1,1))
    self.resblock1=VQ_VAE_ResBlock(out_channels)
    self.resblock2=VQ_VAE_ResBlock(out_channels)
    self.resblock3=VQ_VAE_ResBlock(out_channels)
    self.resblock4=VQ_VAE_ResBlock(out_channels)
    self.norm1=nn.BatchNorm3d(out_channels)
    self.relu1=nn.ReLU()
  def forward(self, x):
    x=self.conv1(x)
    x=self.conv2(x)
    x=self.conv3(x)
    x=self.resblock1(x)
    x=self.resblock2(x)
    x=self.resblock3(x)
    x=self.resblock4(x)
    x=self.norm1(x)
    x=self.relu1(x)
    return x



In [ ]:
levels=[8,5,5,5,5]
class FSQ(nn.Module):
  def __init__(self, levels, n_hidden=240):
    super().__init__()
    self.register_buffer("levels", torch.tensor(levels).reshape(1, len(levels), 1,1,1))

    self.levels=torch.tensor(levels)
    self.levels=self.levels.reshape(1,1,1,len(levels),1)
    self.into_conv=nn.Conv3d(n_hidden, len(levels), kernel_size=(1,1,1), bias=False)
    self.out_conv=nn.Conv3d(len(levels), n_hidden, kernel_size=(1,1,1), bias=False)


  def forward(self, z):
    z=self.into_conv(z)
    zb=(self.levels-1)/2*torch.tanh(z)
    zq=zb+(zb.round()-zb).detach()
    zq=self.out_conv(zq)
    return zq




In [ ]:
class Decoder(nn.Module):
  def __init__(self, in_channels):
    super().__init__()
    self.resblock1=VQ_VAE_ResBlock(in_channels)
    self.resblock2=VQ_VAE_ResBlock(in_channels)
    self.resblock3=VQ_VAE_ResBlock(in_channels)
    self.resblock4=VQ_VAE_ResBlock(in_channels)

    self.norm1=nn.BatchNorm3d(in_channels)
    self.relu1=nn.ReLU()

    self.conv1=nn.ConvTranspose3d(in_channels, in_channels, kernel_size=(3,3,3), stride=(2,1,1), padding=1, output_padding=(1,0,0))
    self.conv2=nn.ConvTranspose3d(in_channels, 3, kernel_size=(3,3,3), stride=(2,2,2), padding=1, output_padding=(1,1,1))
  def forward(self, x):
    x=self.resblock1(x)
    x=self.resblock2(x)
    x=self.resblock3(x)
    x=self.resblock4(x)

    x=self.norm1(x)
    x=self.relu1(x)

    x=self.conv1(x)
    x=self.conv2(x)
    x = torch.tanh(x)
    return x

In [ ]:
class VAE_FSQ(nn.Module):
  def __init__(self, n_hidden, levels):

    super().__init__()
    self.encoder=Encoder(n_hidden)
    self.fsq=FSQ(levels, n_hidden)
    self.decoder=Decoder(n_hidden)
  def forward(self, x):
    h=self.encoder(x)
    h=self.fsq(h)
    h=self.decoder(h)
    return h, (F.mse_loss(x, h)/(0.06))